# Hydris PS1 - GEE Pilot  (v2)
**Goal:** prove PS1's data backbone with *zero heavy storage*.

Given a factory coordinate, get its **watershed basin** + a **multi-risk profile**
(scarcity / flood / drought / groundwater), **today and 2050**, straight from
**WRI Aqueduct 4.0** hosted in **Google Earth Engine**.

This notebook is the reference implementation of pipeline stages **S1 (basin resolve)**
and **S2 (risk join + future)**. Once it works here, the same logic gets wrapped into
the FastAPI backend.

**v2 fixes baked in:** `-9999` no-data sanitizing, a `future_stress()` 2050 pull, and a
map basemap that isn't blocked. Run cells top to bottom; edit the two `# <-- EDIT` lines.


## 1. Install + imports

In [ ]:
!pip install -q earthengine-api geemap geopandas folium
import ee, re, json
import pandas as pd
print("libs ready")


## 2. Authenticate Earth Engine
Earth Engine needs a free **Google Cloud project** (make one at
https://code.earthengine.google.com). No card, noncommercial = free.

In [ ]:
EE_PROJECT = "your-gee-project-id"   # <-- EDIT: your Earth Engine Cloud project id

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)
print("Earth Engine initialized")


## 3. Pilot sites
Five real locations chosen to exercise different risk regimes - including no-data and arid edge cases.

In [ ]:
SITES = [
    {"id": "S1", "name": "Chennai plant (IN)",    "lat": 13.0827, "lng": 80.2707},   # high stress
    {"id": "S2", "name": "Tiruppur textile (IN)", "lat": 11.1085, "lng": 77.3411},   # textile hub
    {"id": "S3", "name": "Fresno plant (US-CA)",  "lat": 36.7378, "lng": -119.7871}, # groundwater depletion
    {"id": "S4", "name": "Hamburg plant (DE)",    "lat": 53.5511, "lng": 9.9937},    # low risk
    {"id": "S5", "name": "Riyadh plant (SA)",     "lat": 24.7136, "lng": 46.6753},   # arid edge case
]  # <-- EDIT to your own sites
print(len(SITES), "sites loaded")


## 4. Load Aqueduct 4.0 collections
`baseline_annual` = today's 13 indicators. `future_annual` = 2030/2050/2080 projections.

In [ ]:
AQ     = ee.FeatureCollection("WRI/Aqueduct_Water_Risk/V4/baseline_annual")
AQ_FUT = ee.FeatureCollection("WRI/Aqueduct_Water_Risk/V4/future_annual")
print("baseline props:", AQ.first().propertyNames().size().getInfo(),
      "| future props:", AQ_FUT.first().propertyNames().size().getInfo())


## 5. Helpers: sanitize + basin query  (this IS stages S1 + S2)
`clean()` turns Aqueduct's `-9999` no-data marker into `None` so it's never shown or averaged.
`risk_at()` does point-in-polygon **inside GEE** and returns the basin's properties.

In [ ]:
IND = {                       # PS1 risk dimension -> Aqueduct score column
    "scarcity":    "bws_score", # baseline water stress
    "flood":       "rfr_score", # riverine flood risk
    "drought":     "drr_score", # drought risk
    "groundwater": "gtd_score", # groundwater table decline
}
OVERALL     = "w_awr_def_tot_score"   # default-weighted overall
OVERALL_CAT = "w_awr_def_tot_cat"

def clean(v):
    """Aqueduct uses -9999 for 'no data' -> convert to None (never a real score)."""
    return None if v in (-9999, -9999.0, "-9999") else v

def risk_at(lat, lng, fc=AQ):
    """All Aqueduct properties for the basin containing (lat,lng); None if no basin."""
    pt  = ee.Geometry.Point([lng, lat])          # order is [lng, lat]
    hit = fc.filterBounds(pt)
    if hit.size().getInfo() == 0:
        return None
    return hit.first().toDictionary().getInfo()

results = {}
for s in SITES:
    p = risk_at(s["lat"], s["lng"])
    results[s["id"]] = p
    print(f'{s["id"]}  {s["name"]:<22}  basin={"NO BASIN" if p is None else p.get("pfaf_id")}')


## 6. Verify our expected columns exist (guard against naming drift)

In [ ]:
if results.get("S1"):
    have = set(results["S1"].keys())
    for label, col in {**IND, "overall": OVERALL, "overall_cat": OVERALL_CAT}.items():
        print(("OK     " if col in have else "MISSING"), label, "->", col)


## 7. Future projections: the field grammar
Future fields are named **`{scenario}{year}_{variable}_x_{stat}`**:
- scenario: `opt` (SSP1-2.6) / `bau` (SSP3-7.0) / `pes` (SSP5-8.5)
- year: `30` / `50` / `80`  |  variable: `ws` stress, `wd` depletion, `iv`/`sv` variability
- stat: `s` score / `c` cat / `r` raw / `l` label

**Important:** only **stress/depletion/variability** are projected - NOT flood/drought/groundwater.
So the 2050 toggle applies to **scarcity**; the other three stay at baseline (by design).

In [ ]:
SCN = {"optimistic": "opt", "bau": "bau", "pessimistic": "pes"}

def future_stress(pfaf_id, year=50, scenario="bau", fc=AQ_FUT):
    """Projected water-stress (scarcity) score for a basin, by year + scenario."""
    f = fc.filter(ee.Filter.eq("pfaf_id", pfaf_id)).first()
    if f is None:
        return {"score": None, "cat": None}
    s_key, c_key = f"{SCN[scenario]}{year}_ws_x_s", f"{SCN[scenario]}{year}_ws_x_c"
    d = f.toDictionary([s_key, c_key]).getInfo()
    return {"score": clean(d.get(s_key)), "cat": d.get(c_key)}

# demo: every site's 2050 business-as-usual scarcity
for s in SITES:
    p = results[s["id"]] or {}
    pf = p.get("pfaf_id")
    fut = future_stress(pf, year=50, scenario="bau") if pf else {"score": None}
    print(f'{s["name"]:<22} scarcity now={clean(p.get(IND["scarcity"]))}  2050(bau)={fut["score"]}')


## 8. Clean risk table (today + 2050 scarcity)
`-9999` values now show as `None` = "no data" (e.g. Chennai groundwater, Riyadh drought).

In [ ]:
rows = []
for s in SITES:
    p  = results[s["id"]] or {}
    pf = p.get("pfaf_id")
    fut = future_stress(pf, year=50, scenario="bau") if pf else {"score": None}
    rows.append({
        "site": s["name"],
        "basin": pf,
        "scarcity_now":  clean(p.get(IND["scarcity"])),
        "scarcity_2050": fut["score"],
        "flood":         clean(p.get(IND["flood"])),
        "drought":       clean(p.get(IND["drought"])),
        "groundwater":   clean(p.get(IND["groundwater"])),
        "overall":       clean(p.get(OVERALL)),
        "overall_cat":   p.get(OVERALL_CAT),
    })
df = pd.DataFrame(rows)
df.round(2)


## 9. Export a tiny fallback file
`pilot_basins.geojson` = your 5 basins (simplified polygons) + cleaned risk attributes.
A few KB - feeds the local pipeline and is your offline safety net during the live demo.

In [ ]:
KEEP = ["pfaf_id","bws_score","bwd_score","gtd_score","drr_score","rfr_score","cfr_score",
        OVERALL, OVERALL_CAT]

features = []
for s in SITES:
    pt  = ee.Geometry.Point([s["lng"], s["lat"]])
    hit = AQ.filterBounds(pt)
    if hit.size().getInfo() == 0:
        continue
    basin = hit.first()
    geom  = basin.geometry().simplify(maxError=1000).getInfo()
    pr    = {k: clean(v) for k, v in basin.toDictionary(KEEP).getInfo().items()}
    pr.update({"site_id": s["id"], "name": s["name"], "lat": s["lat"], "lng": s["lng"]})
    features.append({"type": "Feature", "geometry": geom, "properties": pr})

with open("pilot_basins.geojson", "w") as f:
    json.dump({"type": "FeatureCollection", "features": features}, f)
print("wrote pilot_basins.geojson (", len(features), "basins )")
from google.colab import files
files.download("pilot_basins.geojson")


## 10. Quick visual check (CartoDB basemap - not blocked)

In [ ]:
import folium
m = folium.Map(location=[20, 30], zoom_start=2, tiles="CartoDB positron")
for s in SITES:
    p = results[s["id"]] or {}
    folium.CircleMarker(
        [s["lat"], s["lng"]], radius=7, color="crimson", fill=True,
        popup=f'{s["name"]}  overall={clean(p.get(OVERALL))} ({p.get(OVERALL_CAT)})'
    ).add_to(m)
m


---
**What this proved:** stages S1 (basin resolve) + S2 (risk join + future) run entirely in
Earth Engine, no local storage, with no-data handled safely and a real 2050 scarcity pathway.

**Data:** WRI Aqueduct 4.0 (`WRI/Aqueduct_Water_Risk/V4/*`) via Google Earth Engine - free with attribution to WRI.

**Next:** wrap `risk_at()` + `future_stress()` in a FastAPI endpoint, and run our own weighted
aggregation over the `*_score` columns for the live re-weighting sliders.